In [1]:
!pip install -q -U transformers accelerate sentencepiece safetensors
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.4 MB/s eta 0:00:00
True Tesla T4


In [2]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

/kaggle/working/alta2026_pipeline
answer_initial.csv	  infer.py	    thresholds.py
artifacts		  metadata	    train.csv
blend.py		  metrics.py	    transformer_multitask.py
classical_baseline.py	  README.md	    validate_answer.py
classical_baseline_v2.py  requirements.txt  valid.csv
evaluate.py		  run_baseline.sh


In [3]:
%%writefile idan_model.py
"""
idan_model.py — Incongruity-Aware Dual-Attention Network (IDAN)
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteralCNNBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256, kernel_sizes=(2, 3, 4, 5)):
        super().__init__()
        per_k = out_channels // len(kernel_sizes)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, per_k, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.out_channels = per_k * len(kernel_sizes)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(self.out_channels)

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        x = token_embeds.transpose(1, 2)
        feats = []
        for conv in self.convs:
            f = self.act(conv(x))
            f = f[:, :, :token_embeds.size(1)]
            feats.append(f)
        out = torch.cat(feats, dim=1)
        out = out.transpose(1, 2)
        return self.norm(out)


class LiteralLinearBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256):
        super().__init__()
        self.proj = nn.Linear(embed_dim, out_channels)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_channels)
        self.out_channels = out_channels

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        return self.norm(self.act(self.proj(token_embeds)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, channels),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        avg_pool = (x * m).sum(dim=1) / denom
        max_pool = (x.masked_fill(m == 0, float("-inf"))).max(dim=1).values
        channel_att = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * channel_att.unsqueeze(1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        avg_pool = x.mean(dim=2, keepdim=True)
        max_pool = x.max(dim=2, keepdim=True).values
        pooled = torch.cat([avg_pool, max_pool], dim=2).transpose(1, 2)
        att = torch.sigmoid(self.conv(pooled)).transpose(1, 2)
        att = att * mask.unsqueeze(-1)
        return x * att


class IncongruityDualAttention(nn.Module):
    def __init__(self, literal_dim: int, contextual_dim: int, proj_dim: int = 384,
                 use_channel_attention: bool = True, use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.lit_proj = nn.Linear(literal_dim, proj_dim)
        self.ctx_proj = nn.Linear(contextual_dim, proj_dim)
        joint_dim = proj_dim * 2
        self.use_channel_attention = use_channel_attention
        self.use_spatial_attention = use_spatial_attention
        self.use_incongruity_features = use_incongruity_features
        if use_channel_attention:
            self.channel_att = ChannelAttention(joint_dim)
        if use_spatial_attention:
            self.spatial_att = SpatialAttention()
        self.fuse_norm = nn.LayerNorm(joint_dim)
        self.proj_dim = proj_dim

    def output_dim(self) -> int:
        return self.proj_dim * 2 + (self.proj_dim * 2 if self.use_incongruity_features else 0)

    def forward(self, literal_feats, contextual_feats, mask):
        lit = self.lit_proj(literal_feats)
        ctx = self.ctx_proj(contextual_feats)
        joint = torch.cat([lit, ctx], dim=-1)

        if self.use_channel_attention:
            joint = self.channel_att(joint, mask)
        if self.use_spatial_attention:
            joint = self.spatial_att(joint, mask)
        joint = self.fuse_norm(joint)

        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        pooled_fused = (joint * m).sum(dim=1) / denom

        if not self.use_incongruity_features:
            return pooled_fused

        lit_pooled = (lit * m).sum(dim=1) / denom
        ctx_pooled = (ctx * m).sum(dim=1) / denom
        diff = lit_pooled - ctx_pooled
        prod = lit_pooled * ctx_pooled

        return torch.cat([pooled_fused, diff, prod], dim=-1)


class IDAN(nn.Module):
    def __init__(self, encoder, hidden_size: int, cnn_out_channels: int = 256,
                 proj_dim: int = 384, dropout: float = 0.15,
                 literal_encoder: str = "cnn",
                 use_channel_attention: bool = True,
                 use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.encoder = encoder
        self.embed_layer = encoder.get_input_embeddings()

        if literal_encoder == "cnn":
            self.literal_branch = LiteralCNNBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        elif literal_encoder == "linear":
            self.literal_branch = LiteralLinearBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        else:
            raise ValueError(f"unknown literal_encoder: {literal_encoder}")

        self.fusion = IncongruityDualAttention(
            literal_dim=self.literal_branch.out_channels,
            contextual_dim=hidden_size,
            proj_dim=proj_dim,
            use_channel_attention=use_channel_attention,
            use_spatial_attention=use_spatial_attention,
            use_incongruity_features=use_incongruity_features,
        )
        fused_dim = self.fusion.output_dim()
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))
        self.sarc_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        enc_out = self.encoder(**kwargs)
        contextual_feats = enc_out.last_hidden_state

        token_embeds = self.embed_layer(input_ids)
        literal_feats = self.literal_branch(token_embeds)

        mask = attention_mask.float()
        fused = self.fusion(literal_feats, contextual_feats, mask)
        fused = self.drop(fused)
        return self.sent_head(fused), self.sarc_head(fused)

Writing idan_model.py


In [4]:
%%writefile idan_train.py
"""
idan_train.py — trains IDAN (Incongruity-Aware Dual-Attention Network).
"""
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import alta_score, competition_stratify_key
from transformer_multitask import SPECIAL_TOKENS, TextDataset
from idan_model import IDAN


@dataclass
class IDANConfig:
    backbone: str = "microsoft/deberta-v3-base"
    max_length: int = 192
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    cnn_out_channels: int = 256
    proj_dim: int = 384
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    gradient_accumulation_steps: int = 2
    seed: int = 42
    n_folds: int = 3
    early_stopping_patience: int = 2
    literal_encoder: str = "cnn"
    use_channel_attention: bool = True
    use_spatial_attention: bool = True
    use_incongruity_features: bool = True
    max_sarcasm_class_weight: float = 5.0


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_model(cfg: IDANConfig, n_new_tokens: int):
    encoder = AutoModel.from_pretrained(cfg.backbone, torch_dtype=torch.float32)
    encoder.resize_token_embeddings(encoder.config.vocab_size + n_new_tokens)
    model = IDAN(
        encoder=encoder,
        hidden_size=encoder.config.hidden_size,
        cnn_out_channels=cfg.cnn_out_channels,
        proj_dim=cfg.proj_dim,
        dropout=cfg.dropout,
        literal_encoder=cfg.literal_encoder,
        use_channel_attention=cfg.use_channel_attention,
        use_spatial_attention=cfg.use_spatial_attention,
        use_incongruity_features=cfg.use_incongruity_features,
    )
    return model


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
    return np.concatenate(ps), np.concatenate(pz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = build_model(cfg, n_new_tokens=len(SPECIAL_TOKENS)).to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * cfg.warmup_ratio), steps)

    s_loss = nn.CrossEntropyLoss()
    n_pos = max(int(train_df["sarcasm"].sum()), 1)
    n_neg = max(len(train_df) - n_pos, 1)
    pos_weight = min(n_neg / n_pos, cfg.max_sarcasm_class_weight)
    z_weight = torch.tensor([1.0, pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    print(f"[IDAN] fold={fold}: sarcasm class weight = {pos_weight:.2f} (n_pos={n_pos}, n_neg={n_neg})")
    best, best_state, epochs_since_improve = -1.0, None, 0

    swa_state = None
    swa_count = 0
    SWA_TOLERANCE = 0.01

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")

        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if comp >= best - SWA_TOLERANCE:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = current_state
                swa_count = 1
            else:
                swa_count += 1
                for k in swa_state:
                    if swa_state[k].dtype.is_floating_point:
                        swa_state[k] = swa_state[k] + (current_state[k] - swa_state[k]) / swa_count

        if epochs_since_improve >= cfg.early_stopping_patience:
            print(f"[IDAN] fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
            break

    if swa_state is not None and swa_count > 1:
        model.load_state_dict(swa_state)
        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        _, swa_comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold}: SWA (n={swa_count} epochs) score={swa_comp:.5f} vs best-single-epoch={best:.5f}")
        if swa_comp > best:
            print(f"[IDAN] fold={fold}: SWA wins, using averaged weights")
            best = swa_comp
            best_state = swa_state
        else:
            model.load_state_dict(best_state)

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    with open(out_dir / f"fold{fold}.json", "w") as f:
        json.dump({"fold": fold, "best_validation_score": best, "config": asdict(cfg)}, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv, out_dir, cfg: IDANConfig, external_valid_csv=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)

        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            eps, epz = evaluate(model, ext_loader, device)
            ext_ps_all.append(eps); ext_pz_all.append(epz)

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("[IDAN] OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("[IDAN] External valid @0.5", ext_scores, ext_comp)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/idan")
    ap.add_argument("--backbone", default="microsoft/deberta-v3-base")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-length", type=int, default=192)
    ap.add_argument("--n-folds", type=int, default=3)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--literal-encoder", choices=["cnn", "linear"], default="cnn")
    ap.add_argument("--no-channel-attention", action="store_true")
    ap.add_argument("--no-spatial-attention", action="store_true")
    ap.add_argument("--no-incongruity-features", action="store_true")
    ap.add_argument("--no-class-weight", action="store_true")
    args, _unknown = ap.parse_known_args()

    cfg = IDANConfig(
        backbone=args.backbone, epochs=args.epochs, batch_size=args.batch_size,
        max_length=args.max_length, n_folds=args.n_folds,
        early_stopping_patience=args.patience, gradient_accumulation_steps=args.grad_accum,
        literal_encoder=args.literal_encoder,
        use_channel_attention=not args.no_channel_attention,
        use_spatial_attention=not args.no_spatial_attention,
        use_incongruity_features=not args.no_incongruity_features,
        max_sarcasm_class_weight=1.0 if args.no_class_weight else 5.0,
    )
    cv_train(args.train, args.out, cfg, args.valid)

Writing idan_train.py


In [5]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_noincong_clean \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2 \
  --no-incongruity-features \
  --no-class-weight

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 2.52MB/s]
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 268kB/s]
spm.model: 100%|███████████████████████████| 2.46M/2.46M [00:00<00:00, 4.13MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100%|██████████████████████| 371M/371M [00:03<00:00, 123MB/s]
Loading weights: 100%|███████████████████████| 198/198 [00:00<00:00, 653.73it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.

In [6]:
from thresholds import optimize_thresholds, save_thresholds, apply_thresholds
from metrics import alta_score
import pandas as pd

valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/idan_noincong_clean/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/idan_noincong_clean/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print('CLEAN NO-INCONGRUITY ABLATION:', scores, final)

CLEAN NO-INCONGRUITY ABLATION: {'sentiment-en-AU': 0.919066317626527, 'sentiment-en-UK': 0.9506310793982027, 'sarcasm-en-AU': 0.7579478054567023, 'sarcasm-en-UK': 0.7232587825904488} 0.8211625501084879
